Загрузка модели и подготовка:

In [5]:
import pandas as pd
import joblib
import ipywidgets as widgets
from IPython.display import display, clear_output

# Загружаем нашу лучшую модель (Linear Regression)
model_path = "../models/best_demand_model.pkl"
bundle = joblib.load(model_path)
model = bundle["model"]
FEATURES = bundle["features"]

print("Модель успешно загружена. Готова к работе!")

Модель успешно загружена. Готова к работе!


Создание элементов интерфейса:

In [ ]:
# Стилизация
style = {'description_width': '250px'}
layout = widgets.Layout(width='500px')

# 1. Календарные признаки
w_year = widgets.IntSlider(value=2018, min=2018, max=2030, step=1, description='Плановый год:', style=style, layout=layout)
w_month = widgets.IntSlider(value=6, min=1, max=12, step=1, description='Плановый месяц:', style=style, layout=layout)

# Автоматический расчет квартала
w_quarter = widgets.Label(value="Квартал: 2")
def update_quarter(change):
    q = (change['new'] - 1) // 3 + 1
    w_quarter.value = f"Квартал: {q}"
w_month.observe(update_quarter, names='value')
update_quarter({'new': 6}) # Инициализация

# 2. Лаги и скользящие средние
w_lag1 = widgets.FloatText(value=600.0, description='Спрос в прошлом месяце (lag_1):', style=style, layout=layout)
w_lag2 = widgets.FloatText(value=580.0, description='Спрос 2 месяца назад (lag_2):', style=style, layout=layout)
w_lag3 = widgets.FloatText(value=550.0, description='Спрос 3 месяца назад (lag_3):', style=style, layout=layout)
w_lag12 = widgets.FloatText(value=450.0, description='Спрос год назад (lag_12):', style=style, layout=layout)

w_roll_mean = widgets.FloatText(value=576.0, description='Скользящее среднее (3 мес):', style=style, layout=layout)
w_roll_std = widgets.FloatText(value=25.0, description='Скользящее ст. отклонение:', style=style, layout=layout)

# Кнопка запуска прогноза
btn_predict = widgets.Button(description="Рассчитать прогноз", button_style='success', layout=widgets.Layout(width='300px', height='40px'))
out = widgets.Output() # Место для вывода результата

Логика работы и отображение интерфейса:

In [7]:
# Функция, которая срабатывает при нажатии на кнопку
def on_predict_clicked(b):
    with out:
        clear_output()
        
        # Собираем данные из виджетов
        input_dict = {
            "lag_1": w_lag1.value,
            "lag_2": w_lag2.value,
            "lag_3": w_lag3.value,
            "lag_12": w_lag12.value,
            "rolling_mean_3": w_roll_mean.value,
            "rolling_std_3": w_roll_std.value,
            "month": w_month.value,
            "quarter": (w_month.value - 1) // 3 + 1, # Пересчитываем на всякий случай
            "year": w_year.value
        }
        
        # Превращаем в DataFrame, строго соблюдая порядок признаков (FEATURES)
        df_input = pd.DataFrame([input_dict])[FEATURES]
        
        # Делаем прогноз
        prediction = model.predict(df_input)[0]
        
        # Выводим результат
        print("=" * 50)
        print(f"ПРОГНОЗ СПРОСА: {prediction:,.0f} тонн")
        print("=" * 50)
        print("Прогноз может быть использован специалистом ПЭУ для формирования MPS.")

# Привязываем клик к кнопке
btn_predict.on_click(on_predict_clicked)

# Собираем интерфейс: группируем виджеты
header = widgets.HTML("<h3>Введите параметры для прогноза:</h3>")
calendar_box = widgets.VBox([w_year, w_month, w_quarter])
history_box = widgets.VBox([w_lag1, w_lag2, w_lag3, w_lag12, w_roll_mean, w_roll_std])

ui = widgets.VBox([
    header,
    widgets.HTML("<b>1. Календарный план</b>"),
    calendar_box,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>2. История спроса (тонн)</b>"),
    history_box,
    widgets.HTML("<hr>"),
    btn_predict,
    out
])

# Отображаем интерфейс на экране
display(ui)